# 📊 Exercícios — Machine Learning (Aprendizado de Máquina)

**Disciplina:** Inteligência Artificial | **Nível:** Intermediário

> Pratique os fundamentos de ML: regressão linear, KNN e avaliação de modelos.


## 1. Regressão Linear do Zero

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dataset sintético
np.random.seed(42)
X = 2 * np.random.rand(60, 1)
y = 3 + 2.5 * X.squeeze() + np.random.randn(60) * 0.5

# Adicionar coluna de 1s para o viés
X_b = np.c_[np.ones(len(X)), X]

# Solução analítica (Equação Normal)
theta_best = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
print(f"Parâmetros (Equação Normal): θ₀={theta_best[0]:.3f}, θ₁={theta_best[1]:.3f}")
print(f"Valores reais:               θ₀=3.000,  θ₁=2.500")

# Previsão
X_plot = np.array([[0],[2]])
X_plot_b = np.c_[np.ones(2), X_plot]
y_plot = X_plot_b @ theta_best

plt.figure(figsize=(8,4))
plt.scatter(X, y, alpha=0.6, label='Dados de treinamento')
plt.plot(X_plot, y_plot, 'r-', linewidth=2, label=f'Modelo: y={theta_best[0]:.2f}+{theta_best[1]:.2f}x')
plt.xlabel('X'); plt.ylabel('y'); plt.title('Regressão Linear'); plt.legend(); plt.grid(True); plt.show()


### 📝 Exercício 1

Implemente a regressão linear usando **Gradiente Descendente** (sem a equação normal). Treine por 1000 épocas com taxa de aprendizado 0.1 e compare os parâmetros θ encontrados.

In [ ]:
def gradiente_descendente(X_b, y, eta=0.1, n_epocas=1000):
    """TODO: implemente o gradiente descendente."""
    m = len(y)
    theta = np.random.randn(2)  # θ aleatório
    historico = []
    for epoca in range(n_epocas):
        y_pred = X_b @ theta
        erro = y_pred - y
        gradiente = (2/m) * X_b.T @ erro
        theta -= eta * gradiente
        historico.append(np.mean(erro**2))  # MSE
    return theta, historico

theta_gd, historico = gradiente_descendente(X_b, y)
print(f"Gradiente Descendente: θ₀={theta_gd[0]:.3f}, θ₁={theta_gd[1]:.3f}")
print(f"Equação Normal:        θ₀={theta_best[0]:.3f}, θ₁={theta_best[1]:.3f}")

plt.plot(historico); plt.xlabel('Época'); plt.ylabel('MSE'); plt.title('Curva de aprendizado'); plt.grid(True); plt.show()


## 2. KNN — K-Vizinhos Mais Próximos

In [ ]:
from collections import Counter

class KNN:
    """KNN implementado do zero."""
    
    def __init__(self, k=3):
        self.k = k
    
    def fit(self, X, y):
        self.X_treino = X
        self.y_treino = y
    
    def _distancia(self, a, b):
        return np.sqrt(np.sum((a-b)**2))
    
    def prever_um(self, x):
        distancias = [(self._distancia(x, xi), yi)
                      for xi, yi in zip(self.X_treino, self.y_treino)]
        distancias.sort(key=lambda d: d[0])
        k_mais_proximos = [yi for _, yi in distancias[:self.k]]
        return Counter(k_mais_proximos).most_common(1)[0][0]
    
    def prever(self, X):
        return np.array([self.prever_um(x) for x in X])

# Dataset de flores (Iris simplificado — 2 classes, 2 features)
np.random.seed(42)
X_class = np.r_[
    np.random.randn(30, 2) + [0, 0],    # classe 0
    np.random.randn(30, 2) + [3, 3],    # classe 1
]
y_class = np.array([0]*30 + [1]*30)

# Treino/Teste
idx = np.random.permutation(60)
X_tr, y_tr = X_class[idx[:45]], y_class[idx[:45]]
X_te, y_te = X_class[idx[45:]], y_class[idx[45:]]

resultados = {}
for k in [1, 3, 5, 7, 11]:
    knn = KNN(k=k)
    knn.fit(X_tr, y_tr)
    preds = knn.prever(X_te)
    acc = (preds == y_te).mean()
    resultados[k] = acc
    print(f"K={k:2d} → Acurácia: {acc:.2%}")

melhor_k = max(resultados, key=resultados.get)
print(f"\nMelhor K: {melhor_k} (acurácia={resultados[melhor_k]:.2%})")


### 📝 Exercício 2

Visualize as **fronteiras de decisão** do KNN para K=1 e K=9. Qual parece mais generalizar bem?

In [ ]:
def plotar_fronteira(knn, X, y, titulo):
    h = 0.05
    x_min,x_max = X[:,0].min()-1, X[:,0].max()+1
    y_min,y_max = X[:,1].min()-1, X[:,1].max()+1
    xx,yy = np.meshgrid(np.arange(x_min,x_max,h), np.arange(y_min,y_max,h))
    Z = knn.prever(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.figure(figsize=(6,5))
    plt.contourf(xx,yy,Z,alpha=0.3,cmap='bwr')
    plt.scatter(X[:,0],X[:,1],c=y,cmap='bwr',edgecolors='k',s=50)
    plt.title(titulo); plt.grid(True); plt.show()

for k in [1, 9]:
    knn = KNN(k=k); knn.fit(X_tr, y_tr)
    plotar_fronteira(knn, X_class, y_class, f'Fronteira de decisão KNN (K={k})')


## 3. Métricas de Avaliação

In [ ]:
def calcular_metricas(y_real, y_pred):
    tp = np.sum((y_real==1) & (y_pred==1))
    tn = np.sum((y_real==0) & (y_pred==0))
    fp = np.sum((y_real==0) & (y_pred==1))
    fn = np.sum((y_real==1) & (y_pred==0))
    
    acuracia  = (tp+tn) / len(y_real)
    precisao  = tp/(tp+fp) if (tp+fp)>0 else 0
    recall    = tp/(tp+fn) if (tp+fn)>0 else 0
    f1        = 2*precisao*recall/(precisao+recall) if (precisao+recall)>0 else 0
    
    print(f"Acurácia:  {acuracia:.3f}")
    print(f"Precisão:  {precisao:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1-Score:  {f1:.3f}")
    print(f"Matriz de confusão:")
    print(f"  TP={tp}  FP={fp}")
    print(f"  FN={fn}  TN={tn}")

knn_best = KNN(k=melhor_k); knn_best.fit(X_tr, y_tr)
preds_test = knn_best.prever(X_te)
calcular_metricas(y_te, preds_test)


### 📝 Exercício Final

Gere um dataset **desequilibrado** (80% classe 0, 20% classe 1) e observe como a acurácia pode ser enganosa. Calcule todas as métricas e compare com o F1-Score.

*Dica: um modelo que sempre prevê classe 0 teria 80% de acurácia mas F1=0!*

In [ ]:
np.random.seed(7)
X_imbal = np.r_[np.random.randn(80,2)+[0,0], np.random.randn(20,2)+[4,4]]
y_imbal = np.array([0]*80 + [1]*20)

# Modelo ingênuo: sempre prevê 0
y_pred_ingenuo = np.zeros(100, dtype=int)
print("Modelo ingênuo (sempre prevê 0):")
calcular_metricas(y_imbal, y_pred_ingenuo)

print("\nKNN K=5 no dataset desequilibrado:")
knn5 = KNN(k=5); knn5.fit(X_imbal[:80], y_imbal[:80])
preds_imbal = knn5.prever(X_imbal[80:])
# TODO: calcule as métricas para o KNN
